# Coral Reef Segmentation v2 — fine-tune the published Coralscapes SegFormer (Colab)

**Runtime:** `Runtime → Change runtime type → T4 GPU` (A100/L4 if available: faster, same code).

**What changes vs v1** (`colab_train_coralscapes.ipynb`, SegFormer MiT-B2 from ImageNet at 512×1024):
1. **Starts from the authors' Coralscapes model** `EPFL-ECEO/segformer-b2-finetuned-coralscapes-1024-1024`
   (Apache-2.0; trained on the Coralscapes *train* split only, so the official TEST sites stay unseen). Its 39-class
   head is kept: the 4 groups (other / live / bleached / dead coral) are obtained by adding up the probabilities of
   the member classes, so nothing it learned is thrown away.
2. **Higher resolution:** trains on 768×768 crops from 768×1536 frames; evaluates at the native 1024×2048.
3. **Stronger augmentation** (scale, rotation, colour/brightness/hue, blur, noise) — the paper's best result came from this.
4. **Lower, single learning rate** (already-trained model) → steadier validation curves than v1.
5. **Checkpoint chosen by val coral IoU** (steadier than mIoU, which swings with the rare bleached class).
6. **Test-time augmentation:** prediction averaged with the horizontally flipped image.

**Fair comparison:** the same evaluation code scores (a) the published model with no fine-tuning, (b) v1 and
(c) v2 — on VAL to choose, then on TEST to report. TEST is never used to choose.

**Output** (download from `MyDrive/coralscapes/` into the local project's `models/coral_segmenter/`):
the folder `coralscapes_segformer_v2/` (model) plus `coralscapes_segformer_v2_config.json` and
`coralscapes_segformer_v2_test_metrics.json`.

In [ ]:
!nvidia-smi
!pip install -q 'transformers>=4.40' 'segmentation-models-pytorch>=0.5' albumentations datasets

In [ ]:
# ---- Config -------------------------------------------------------------
HF_CKPT   = 'EPFL-ECEO/segformer-b2-finetuned-coralscapes-1024-1024'
TAG       = 'coralscapes_segformer_v2'
TRAIN_H, TRAIN_W = 768, 1536   # frames resized to this, then random crops
CROP      = 768
EVAL_H, EVAL_W = 1024, 2048    # native Coralscapes resolution
EPOCHS    = 20
PATIENCE  = 6                  # early stop on val coral IoU
BATCH     = 4                  # T4 16 GB at 768x768 with AMP
LR        = 2e-5               # already-trained model: small, single LR
WARMUP    = 1
SEED      = 42
TTA_FLIP  = True
GATE_CORAL_IOU = 0.50
CORAL_PRESENT_FRAC = 0.02
BLEACH_PRESENT_FRAC = 0.005
V1_CKPT = '/content/drive/MyDrive/coralscapes/coralscapes_segformer.pt'   # v1 weights for the comparison (skipped if missing)
# -------------------------------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/coralscapes'
os.makedirs(DRIVE_DIR, exist_ok=True)

## Data and class groups (same grouping as v1)

In [ ]:
import numpy as np, json, random, torch, cv2
import torch.nn.functional as F
from datasets import load_dataset
from huggingface_hub import hf_hub_download

ds = load_dataset('EPFL-ECEO/coralscapes')
ID2NAME = {int(k): v for k, v in json.load(open(hf_hub_download('EPFL-ECEO/coralscapes', 'id2label.json', repo_type='dataset'))).items()}
CLASSES = ['other', 'live_coral', 'bleached_coral', 'dead_coral']
LIVE, BLEACHED, DEAD = 1, 2, 3
CORAL = [LIVE, BLEACHED, DEAD]
IGNORE = 255
K = len(CLASSES)

def group(name):
    if 'clam' in name:
        return 0
    if 'bleached' in name:
        return BLEACHED
    if 'dead' in name:
        return DEAD
    if 'alive' in name or name in ('millepora', 'turbinaria'):
        return LIVE
    return 0

LUT = np.full(256, IGNORE, np.uint8)
for i, n in ID2NAME.items():
    LUT[i] = group(n)
GROUP_OF_ID = [int(LUT[i]) for i in range(1, 40)]          # model channels 1..39 -> group (channel 0 = unlabelled, unused)
for g, c in enumerate(CLASSES):
    print(f'{c:15s} <- {[ID2NAME[i] for i in ID2NAME if LUT[i] == g]}')

In [ ]:
import albumentations as A
from torch.utils.data import Dataset, DataLoader
torch.manual_seed(SEED); random.seed(SEED); np.random.seed(SEED)

MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
train_tf = A.Compose([
    A.Resize(TRAIN_H, TRAIN_W),
    A.RandomScale((-0.25, 0.5), p=0.7),
    A.PadIfNeeded(CROP, CROP, border_mode=cv2.BORDER_REFLECT_101),
    A.RandomCrop(CROP, CROP),
    A.HorizontalFlip(p=0.5),
    A.Rotate(limit=10, border_mode=cv2.BORDER_REFLECT_101, p=0.3),
    A.ColorJitter(0.3, 0.3, 0.3, 0.05, p=0.7),
    A.OneOf([A.GaussianBlur(blur_limit=(3, 5)), A.GaussNoise()], p=0.3),
    A.Normalize(MEAN, STD),
])
eval_tf = A.Compose([A.Resize(EVAL_H, EVAL_W), A.Normalize(MEAN, STD)])

class Coralscapes(Dataset):
    def __init__(self, split, tf):
        self.d, self.tf = split, tf
    def __len__(self):
        return len(self.d)
    def __getitem__(self, i):
        row = self.d[i]
        img = np.array(row['image'].convert('RGB'))
        mask = LUT[np.array(row['label']).astype(np.uint8)]
        o = self.tf(image=img, mask=mask)
        return torch.from_numpy(o['image'].transpose(2, 0, 1)).float(), torch.from_numpy(o['mask']).long()

dl_train = DataLoader(Coralscapes(ds['train'], train_tf), batch_size=BATCH, shuffle=True, num_workers=2, drop_last=True)
dl_val = DataLoader(Coralscapes(ds['validation'], eval_tf), batch_size=1, num_workers=2)
dl_test = DataLoader(Coralscapes(ds['test'], eval_tf), batch_size=1, num_workers=2)

counts = np.zeros(K)
for i in random.Random(SEED).sample(range(len(ds['train'])), 300):
    m = LUT[np.array(ds['train'][i]['label']).astype(np.uint8)]
    counts += np.bincount(m[m != IGNORE], minlength=K)
freq = counts / counts.sum()
W = 1 / np.sqrt(np.maximum(freq, 1e-6)); W = W / W.mean()
print('pixel share:', dict(zip(CLASSES, freq.round(4))), '\nclass weights:', dict(zip(CLASSES, W.round(2))))

## Model: published Coralscapes SegFormer, 39 classes → 4 groups by summing probabilities

In [ ]:
from transformers import SegformerForSemanticSegmentation
device = 'cuda'
model = SegformerForSemanticSegmentation.from_pretrained(HF_CKPT).to(device)
assert model.config.num_labels == 40, model.config.num_labels
GROUP_INDEX = [torch.tensor([c + 1 for c, g in enumerate(GROUP_OF_ID) if g == k], device=device) for k in range(K)]

def grouped_logp(m, x, size):
    # log-probabilities of the 4 groups at `size`: log-softmax over channels 1..39, then log-sum-exp within each group
    logits = m(pixel_values=x).logits[:, 1:]
    logits = F.interpolate(logits.float(), size=size, mode='bilinear', align_corners=False)
    logp = F.log_softmax(logits, dim=1)
    return torch.stack([torch.logsumexp(logp[:, idx - 1], dim=1) for idx in GROUP_INDEX], dim=1)

def predict_hf(m, x, size):
    lp = grouped_logp(m, x, size).exp()
    if TTA_FLIP:
        lp = (lp + grouped_logp(m, x.flip(-1), size).exp().flip(-1)) / 2
    return lp.argmax(1)

## Shared evaluation (pixel confusion matrix + per-image coral / bleaching fractions)

In [ ]:
def iou_from_cm(cm):
    inter = np.diag(cm); union = cm.sum(0) + cm.sum(1) - inter
    return np.where(union > 0, inter / np.maximum(union, 1), np.nan)

def coral_iou(cm):
    c = np.array(CORAL)
    tp = cm[np.ix_(c, c)].sum(); fp = cm[:, c].sum() - tp; fn = cm[c, :].sum() - tp
    return tp / max(1, tp + fp + fn), tp / max(1, tp + fp), tp / max(1, tp + fn)

def evaluate(predict, loader):
    cm = torch.zeros(K, K, dtype=torch.long, device=device)
    per_image = []
    with torch.no_grad(), torch.autocast('cuda'):
        for x, y in loader:
            y = y.to(device); p = predict(x.to(device), y.shape[-2:])
            valid = y != IGNORE
            cm += torch.bincount(y[valid] * K + p[valid], minlength=K * K).view(K, K)
            for yi, pi, vi in zip(y, p, valid):
                n = vi.sum().clamp(min=1)
                per_image.append([float(((yi == c) & vi).sum() / n) for c in CORAL] + [float(((pi == c) & vi).sum() / n) for c in CORAL])
    return cm.cpu().numpy(), np.array(per_image)

def image_level(true_frac, pred_frac, thr):
    t, p = true_frac >= thr, pred_frac >= thr
    tp, fp, fn = int((t & p).sum()), int((~t & p).sum()), int((t & ~p).sum())
    return {'accuracy': round(float((t == p).mean()), 4), 'precision': round(tp / max(1, tp + fp), 4),
            'recall': round(tp / max(1, tp + fn), 4), 'n_positive': int(t.sum()), 'n_images': len(t),
            'always_majority_baseline': round(float(max(t.mean(), 1 - t.mean())), 4)}

def summarize(cm, img, name, split):
    ious = iou_from_cm(cm); c_iou, c_prec, c_rec = coral_iou(cm); c = np.array(CORAL)
    out = {'model': name, 'split': split, 'pixel_accuracy': round(float(np.diag(cm).sum() / cm.sum()), 4),
           'mean_iou': round(float(np.nanmean(ious)), 4),
           'per_class_iou': {k: (None if np.isnan(v) else round(float(v), 4)) for k, v in zip(CLASSES, ious)},
           'coral_iou': round(float(c_iou), 4), 'coral_precision': round(float(c_prec), 4), 'coral_recall': round(float(c_rec), 4),
           'coral_vs_rest_pixel_accuracy': round(float((cm[np.ix_(c, c)].sum() + cm[0, 0]) / cm.sum()), 4),
           'image_level_coral_present': image_level(img[:, 0:3].sum(1), img[:, 3:6].sum(1), CORAL_PRESENT_FRAC),
           'image_level_bleaching_present': image_level(img[:, 1], img[:, 4], BLEACH_PRESENT_FRAC)}
    print(f"{name:32s} {split:5s} mIoU {out['mean_iou']:.3f}  coral IoU {out['coral_iou']:.3f}  live {out['per_class_iou']['live_coral']:.3f}  "
          f"bleached {out['per_class_iou']['bleached_coral']:.3f}  dead {out['per_class_iou']['dead_coral']:.3f}")
    return out

model.eval()
zero_val = summarize(*evaluate(lambda x, s: predict_hf(model, x, s), dl_val), 'published (no fine-tune)', 'val')

## Fine-tune — weighted NLL on the grouped probabilities + Dice, best checkpoint by val coral IoU

In [ ]:
import segmentation_models_pytorch as smp
dice = smp.losses.DiceLoss('multiclass', ignore_index=IGNORE)      # softmax of log-probs = the probs themselves
nll = torch.nn.NLLLoss(weight=torch.tensor(W, dtype=torch.float32, device=device), ignore_index=IGNORE)
opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
sched = torch.optim.lr_scheduler.SequentialLR(opt, [
    torch.optim.lr_scheduler.LinearLR(opt, start_factor=0.1, total_iters=WARMUP),
    torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS - WARMUP)], milestones=[WARMUP])
scaler = torch.amp.GradScaler('cuda')

CKPT_DIR = f'{DRIVE_DIR}/{TAG}'
best, bad = zero_val['coral_iou'], 0          # must beat the published model as-is, else it is kept unchanged
model.save_pretrained(CKPT_DIR)
print(f'start: published model val coral IoU {best:.3f}')
for epoch in range(EPOCHS):
    model.train(); total = 0
    for x, y in dl_train:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        with torch.autocast('cuda'):
            lp = grouped_logp(model, x, y.shape[-2:])
            loss = nll(lp, y) + dice(lp, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        total += loss.item()
    sched.step()
    model.eval()
    cm, _ = evaluate(lambda x, s: predict_hf(model, x, s), dl_val)
    ious = iou_from_cm(cm); ciou = coral_iou(cm)[0]
    print(f'epoch {epoch+1:3d}  loss {total/len(dl_train):.3f}  val mIoU {np.nanmean(ious):.3f}  coral IoU {ciou:.3f}  '
          f'live {ious[LIVE]:.3f}  bleached {ious[BLEACHED]:.3f}  dead {ious[DEAD]:.3f}', flush=True)
    if ciou > best:
        best, bad = ciou, 0
        model.save_pretrained(CKPT_DIR)
    else:
        bad += 1
        if bad >= PATIENCE:
            print('early stop'); break
print('best val coral IoU', round(best, 3))
model = SegformerForSemanticSegmentation.from_pretrained(CKPT_DIR).to(device).eval()

## Choose on VAL, then report TEST once — published model vs v1 vs v2

In [ ]:
candidates = {'published (no fine-tune)': lambda x, s: predict_hf(published, x, s),
              'v2 fine-tuned': lambda x, s: predict_hf(model, x, s)}
published = SegformerForSemanticSegmentation.from_pretrained(HF_CKPT).to(device).eval()

if os.path.exists(V1_CKPT):
    v1 = smp.create_model('Segformer', encoder_name='mit_b2', encoder_weights=None, classes=K).to(device)
    v1.load_state_dict(torch.load(V1_CKPT, map_location=device)); v1.eval()
    def predict_v1(x, size):
        xs = F.interpolate(x, size=(512, 1024), mode='bilinear', align_corners=False)   # v1 input size
        return F.interpolate(v1(xs).float(), size=size, mode='bilinear', align_corners=False).argmax(1)
    candidates['v1 (ImageNet start, 512x1024)'] = predict_v1
else:
    print('v1 weights not found at', V1_CKPT, '- comparing published vs v2 only')

val_res = {n: summarize(*evaluate(f, dl_val), n, 'val') for n, f in candidates.items()}
chosen = max(val_res, key=lambda n: val_res[n]['coral_iou'])
print('\nchosen on VAL coral IoU:', chosen, '\n')
test_res = {n: summarize(*evaluate(f, dl_test), n, 'test') for n, f in candidates.items()}

In [ ]:
m = dict(test_res[chosen])
m.update(dataset='Coralscapes TEST (official spatial split, 5 unseen dive sites)', chosen_on='val coral_iou',
         eval_hw=[EVAL_H, EVAL_W], tta_flip=TTA_FLIP, n_test_images=len(ds['test']), classes=CLASSES,
         gate_coral_iou=GATE_CORAL_IOU, passed_gate=bool(m['coral_iou'] >= GATE_CORAL_IOU),
         reef_iou=m['coral_iou'], reef_precision=m['coral_precision'], reef_recall=m['coral_recall'],
         all_candidates_test=test_res, all_candidates_val=val_res,
         note='Grouped 4-class task; not comparable to the paper 39-class mIoU.')
print(json.dumps({k: v for k, v in m.items() if not k.startswith('all_')}, indent=2))
json.dump(m, open(f'{DRIVE_DIR}/{TAG}_test_metrics.json', 'w'), indent=2)

if chosen.startswith('v1'):
    print('v1 is better on VAL: keep the v1 files (coralscapes_segformer.*) locally; v2 files are not needed.')
else:
    if chosen.startswith('published'):
        published.save_pretrained(CKPT_DIR)          # fine-tuning did not help: ship the published weights
    json.dump({'backend': 'hf_segformer', 'hf_dir': TAG, 'source': HF_CKPT, 'model': chosen,
               'classes': CLASSES, 'group_of_channel': [None] + GROUP_OF_ID, 'reef_classes': CORAL,
               'live_class': LIVE, 'bleached_class': BLEACHED, 'dead_class': DEAD,
               'input_hw': [EVAL_H, EVAL_W], 'mean': MEAN, 'std': STD, 'tta_flip': TTA_FLIP, 'dataset': 'Coralscapes'},
              open(f'{DRIVE_DIR}/{TAG}_config.json', 'w'), indent=2)
    print(f'Download the folder {TAG}/ and {TAG}_config.json + {TAG}_test_metrics.json from MyDrive/coralscapes '
          'into models/coral_segmenter/.')